In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYC Taxi Analytics Assignment")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 23:36:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
yellow_raw = spark.read.parquet("/home/ec2-user/taxi_data/yellow_tripdata_2026-01.parquet")
green_raw = spark.read.parquet("/home/ec2-user/taxi_data/green_tripdata_2026-01.parquet")

In [3]:
yellow_raw.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [4]:
green_raw.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
yellow_raw.show(5, truncate=False)

[Stage 2:>                                                          (0 + 1) / 1]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2       |2026-01-01 00:54:04 |2026-01-01 00:59:37  |1              |0.97         |1         |N                 |239         |238 

In [6]:
green_raw.show(5, truncate=False)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|1       |2026-01-01 00:27:58 |2026-01-01 00:55:16  |N                 |1         |65          |233       

In [7]:
#standardizing schemas

In [8]:
from pyspark.sql import functions as F

yellow = (yellow_raw
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
    .withColumn("taxi_type", F.lit("yellow"))
)

green = (green_raw
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
    .withColumn("taxi_type", F.lit("green"))
)

common_cols = [
    "VendorID", "pickup_datetime", "dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID",
    "store_and_fwd_flag", "PULocationID", "DOLocationID",
    "payment_type", "fare_amount", "extra", "mta_tax",
    "tip_amount", "tolls_amount", "improvement_surcharge",
    "total_amount", "congestion_surcharge", "cbd_congestion_fee",
    "taxi_type"
]

df = yellow.select(common_cols).unionByName(green.select(common_cols))

In [9]:
print(f"Combined rows: {df.count()}")
df.groupBy("taxi_type").count().show()

Combined rows: 3765161


[Stage 7:===================>                                       (1 + 2) / 3]

+---------+-------+
|taxi_type|  count|
+---------+-------+
|   yellow|3724889|
|    green|  40272|
+---------+-------+



In [10]:
df.show(5, truncate=False)

+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+------------------+---------+
|VendorID|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|cbd_congestion_fee|taxi_type|
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+------------------+---------+
|2       |2026-01-01 00:54:04|2026-01-01 00:59:37|1              |0.97         |1         |N                 |239         |238         |1        

In [11]:
#cleaning

In [12]:
df_clean = (df
    # 1. Remove rows with missing pickup or drop-off timestamps
    .dropna(subset=["pickup_datetime", "dropoff_datetime"])
    # 2. Remove trips with non-positive distance
    .filter(F.col("trip_distance") > 0)
    # 3. Remove trips with negative fares
    .filter(F.col("fare_amount") >= 0)
    # 4. Remove trips with negative total amount
    .filter(F.col("total_amount") >= 0)
    # 5. Remove trips with drop-off time before pickup time
    .filter(F.col("dropoff_datetime") > F.col("pickup_datetime"))
    # 6. Remove trips longer than 24 hours
    .filter(
        (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) <= 24 * 3600
    )
)

In [13]:
before = df.count()
after = df_clean.count()
print(f"Before cleaning: {before}")
print(f"After cleaning:  {after}")
print(f"Removed:         {before - after} ({(before - after) / before * 100:.2f}%)")

[Stage 14:===================>                                      (1 + 2) / 3]

Before cleaning: 3765161
After cleaning:  3557037
Removed:         208124 (5.53%)


In [14]:
df_clean.groupBy("taxi_type").count().show()

[Stage 17:======================================>                   (2 + 1) / 3]

+---------+-------+
|taxi_type|  count|
+---------+-------+
|   yellow|3518128|
|    green|  38909|
+---------+-------+



In [15]:
#Required Analytics

In [16]:
#Question 1: Which taxi type had more trips?
trips_by_type = df_clean.groupBy("taxi_type").count().orderBy(F.desc("count"))
trips_by_type.show()

[Stage 20:>                                                         (0 + 2) / 3]

+---------+-------+
|taxi_type|  count|
+---------+-------+
|   yellow|3518128|
|    green|  38909|
+---------+-------+



In [17]:
#Question 2: What was the average fare by taxi type?
avg_fare_by_type = df_clean.groupBy("taxi_type").agg(
    F.round(F.avg("fare_amount"), 2).alias("avg_fare")
)
avg_fare_by_type.show()

[Stage 23:>                                                         (0 + 2) / 3]

+---------+--------+
|taxi_type|avg_fare|
+---------+--------+
|   yellow|   21.08|
|    green|    16.0|
+---------+--------+



In [18]:
#Question 3: What was the average trip distance by taxi type?
avg_dist_by_type = df_clean.groupBy("taxi_type").agg(
    F.round(F.avg("trip_distance"), 2).alias("avg_distance")
)
avg_dist_by_type.show()

[Stage 26:======================================>                   (2 + 1) / 3]

+---------+------------+
|taxi_type|avg_distance|
+---------+------------+
|   yellow|        6.76|
|    green|       12.99|
+---------+------------+



In [19]:
#Question 4: What hour of day had the most pickups?
pickups_by_hour = (df_clean
    .withColumn("hour", F.hour("pickup_datetime"))
    .groupBy("hour")
    .count()
    .orderBy(F.desc("count"))
)
pickups_by_hour.show(24)

[Stage 29:======================================>                   (2 + 1) / 3]

+----+------+
|hour| count|
+----+------+
|  18|232570|
|  17|224277|
|  15|216305|
|  19|211118|
|  20|208072|
|  21|207428|
|  16|206485|
|  14|202317|
|  13|190970|
|  22|190251|
|  12|183146|
|  11|167923|
|  10|158111|
|   9|154087|
|  23|148822|
|   8|146020|
|   7|112027|
|   0|109013|
|   1| 75623|
|   6| 62649|
|   2| 52470|
|   3| 37491|
|   5| 32070|
|   4| 27792|
+----+------+



In [20]:
#Question 5: What percentage of trips were under 2 miles?
total = df_clean.count()
under_2 = df_clean.filter(F.col("trip_distance") < 2).count()
pct = under_2 / total * 100
print(f"Total trips: {total}")
print(f"Trips under 2 miles: {under_2}")
print(f"Percentage: {pct:.2f}%")

Total trips: 3557037
Trips under 2 miles: 1847954
Percentage: 51.95%


In [21]:
#Question 6: What were the top pickup/drop-off location pairs?
top_routes = (df_clean
    .groupBy("PULocationID", "DOLocationID")
    .count()
    .orderBy(F.desc("count"))
)
top_routes.show(10)

[Stage 38:======================================>                   (2 + 1) / 3]

+------------+------------+-----+
|PULocationID|DOLocationID|count|
+------------+------------+-----+
|         237|         236|23273|
|         236|         237|20188|
|         236|         236|16051|
|         237|         237|14815|
|         161|         237| 9969|
|         237|         161| 9155|
|         142|         239| 8734|
|         239|         238| 8523|
|         161|         236| 8250|
|         141|         236| 8217|
+------------+------------+-----+
only showing top 10 rows


In [22]:
#Question 7: How did weekend trips differ from weekday trips?
df_daytype = (df_clean
    .withColumn("dow", F.dayofweek("pickup_datetime"))
    .withColumn("day_type", F.when(F.col("dow").isin(1, 7), "weekend").otherwise("weekday"))
)

df_daytype.groupBy("day_type").agg(
    F.count("*").alias("trip_count"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    F.round(F.avg("total_amount"), 2).alias("avg_total")
).show()

[Stage 41:======================================>                   (2 + 1) / 3]

+--------+----------+------------+--------+---------+
|day_type|trip_count|avg_distance|avg_fare|avg_total|
+--------+----------+------------+--------+---------+
| weekend|   1019650|        6.42|   20.21|    28.09|
| weekday|   2537387|        6.99|   21.36|    30.23|
+--------+----------+------------+--------+---------+



In [23]:
#Question 8: Predict the fare amount using non-fare predictor columns.

In [24]:
#feature preparation
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

df_ml = (df_clean
    .withColumn("hour", F.hour("pickup_datetime"))
    .withColumn("dow", F.dayofweek("pickup_datetime"))
    .withColumn("trip_seconds",
        F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime"))
    .select("trip_distance", "PULocationID", "DOLocationID",
            "passenger_count", "hour", "dow", "trip_seconds", "fare_amount")
    .dropna()
)

print(f"ML dataset rows: {df_ml.count()}")
df_ml.show(5)

ML dataset rows: 2556384
+-------------+------------+------------+---------------+----+---+------------+-----------+
|trip_distance|PULocationID|DOLocationID|passenger_count|hour|dow|trip_seconds|fare_amount|
+-------------+------------+------------+---------------+----+---+------------+-----------+
|         0.97|         239|         238|              1|   0|  5|         333|        7.2|
|          0.9|         163|         162|              0|   0|  5|         343|        7.9|
|          1.4|          43|         237|              0|   0|  5|         533|       10.7|
|         5.58|         142|         209|              4|   0|  5|        2568|       38.7|
|         2.16|          88|         144|              0|   0|  5|         810|       13.5|
+-------------+------------+------------+---------------+----+---+------------+-----------+
only showing top 5 rows


In [25]:
#train/test split
feature_cols = ["trip_distance", "PULocationID", "DOLocationID",
                "passenger_count", "hour", "dow", "trip_seconds"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_assembled = assembler.transform(df_ml)

train, test = df_assembled.randomSplit([0.8, 0.2], seed=42)
print(f"Train rows: {train.count()}")
print(f"Test rows:  {test.count()}")

Train rows: 2045084


[Stage 51:======================================>                   (2 + 1) / 3]

Test rows:  511300


In [26]:
#random forest model
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    numTrees=50,
    maxDepth=8,
    seed=42
)
model = rf.fit(train)

26/05/21 23:59:52 WARN MemoryStore: Not enough space to cache rdd_240_0 in memory! (computed 141.4 MiB so far)
26/05/21 23:59:52 WARN BlockManager: Persisting block rdd_240_0 to disk instead.
26/05/22 00:00:04 WARN MemoryStore: Not enough space to cache rdd_240_0 in memory! (computed 212.1 MiB so far)
26/05/22 00:00:14 WARN MemoryStore: Not enough space to cache rdd_240_0 in memory! (computed 212.1 MiB so far)
26/05/22 00:00:27 WARN MemoryStore: Not enough space to cache rdd_240_0 in memory! (computed 212.1 MiB so far)
26/05/22 00:00:41 WARN MemoryStore: Not enough space to cache rdd_240_0 in memory! (computed 212.1 MiB so far)
26/05/22 00:00:58 WARN MemoryStore: Not enough space to cache rdd_240_0 in memory! (computed 212.1 MiB so far)
26/05/22 00:01:17 WARN MemoryStore: Not enough space to cache rdd_240_0 in memory! (computed 212.1 MiB so far)
26/05/22 00:01:40 WARN DAGScheduler: Broadcasting large task binary with size 1080.2 KiB
26/05/22 00:01:41 WARN MemoryStore: Not enough space 

In [27]:
#train test rmse 
train_pred = model.transform(train)
test_pred = model.transform(test)

evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="rmse")

train_rmse = evaluator.evaluate(train_pred)
test_rmse = evaluator.evaluate(test_pred)

print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE:  {test_rmse:.4f}")

[Stage 75:======================================>                   (2 + 1) / 3]

Train RMSE: 6.4693
Test RMSE:  6.8148


In [28]:
#feature importance
print("Feature Importances:")
for name, imp in zip(feature_cols, model.featureImportances):
    print(f"  {name}: {imp:.4f}")

Feature Importances:
  trip_distance: 0.4955
  PULocationID: 0.0657
  DOLocationID: 0.0423
  passenger_count: 0.0032
  hour: 0.0025
  dow: 0.0006
  trip_seconds: 0.3901


In [29]:
#predicted vs actual fare
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pdf = test_pred.select("fare_amount", "prediction").sample(0.01, seed=42).toPandas()

plt.figure(figsize=(8, 6))
plt.scatter(pdf["fare_amount"], pdf["prediction"], alpha=0.3, s=5)
plt.xlabel("Actual Fare ($)")
plt.ylabel("Predicted Fare ($)")
plt.title("Predicted vs Actual Fare Amount")
max_val = max(pdf["fare_amount"].max(), pdf["prediction"].max())
plt.plot([0, max_val], [0, max_val], 'r--', label="Perfect prediction")
plt.legend()
plt.tight_layout()
plt.savefig("fare_prediction_plot.png", dpi=150)
plt.show()
print("Plot saved as fare_prediction_plot.png")

Plot saved as fare_prediction_plot.png


In [30]:
#Writing results to s3

In [32]:
trips_by_type.toPandas().to_csv("/home/ec2-user/trips_by_type.csv", index=False)
avg_fare_by_type.toPandas().to_csv("/home/ec2-user/avg_fare_by_type.csv", index=False)
pickups_by_hour.toPandas().to_csv("/home/ec2-user/pickups_by_hour.csv", index=False)
print("Saved locally.")

[Stage 88:>                                                         (0 + 2) / 3]

Saved locally.


In [33]:
import os
os.system("aws s3 cp /home/ec2-user/trips_by_type.csv s3://yegon-de300-hw3/nyc-taxi-assignment/")
os.system("aws s3 cp /home/ec2-user/avg_fare_by_type.csv s3://yegon-de300-hw3/nyc-taxi-assignment/")
os.system("aws s3 cp /home/ec2-user/pickups_by_hour.csv s3://yegon-de300-hw3/nyc-taxi-assignment/")
os.system("aws s3 cp /home/ec2-user/fare_prediction_plot.png s3://yegon-de300-hw3/nyc-taxi-assignment/")
print("Uploaded to S3.")

upload: ./trips_by_type.csv to s3://yegon-de300-hw3/nyc-taxi-assignment/trips_by_type.csv
upload: ./avg_fare_by_type.csv to s3://yegon-de300-hw3/nyc-taxi-assignment/avg_fare_by_type.csv
upload: ./pickups_by_hour.csv to s3://yegon-de300-hw3/nyc-taxi-assignment/pickups_by_hour.csv
upload: ./fare_prediction_plot.png to s3://yegon-de300-hw3/nyc-taxi-assignment/fare_prediction_plot.png
Uploaded to S3.


In [34]:
os.system("aws s3 ls s3://yegon-de300-hw3/nyc-taxi-assignment/")

2026-05-22 00:16:31         43 avg_fare_by_type.csv
2026-05-22 00:16:32      76070 fare_prediction_plot.png
2026-05-22 00:16:31        235 pickups_by_hour.csv
2026-05-22 00:16:30         43 trips_by_type.csv


0